# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [20]:
!pip install duckdb --quiet

import duckdb
from google.colab import userdata

con = duckdb.connect()
hf_token = userdata.get('HF_TOKEN')
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

rel = "hf://datasets/FlyRank/internship-warehouse"

# Confirm connection + get real column names before assuming anything
df_schema = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet') LIMIT 1").df()
for name, dtype in zip(df_schema['column_name'], df_schema['column_type']):
    print(f"{name:<25} {dtype}")

report_date               DATE
client_hash_id            VARCHAR
content_hash_id           VARCHAR
client_has_gsc            BOOLEAN
client_has_ga4            BOOLEAN
gsc_data_available        BOOLEAN
ga4_data_available        BOOLEAN
gsc_impressions           BIGINT
gsc_clicks                BIGINT
gsc_sum_position          BIGINT
gsc_avg_position          DOUBLE
ga4_pageviews             BIGINT
ga4_sessions              BIGINT
ga4_users                 BIGINT
ga4_engaged_sessions      BIGINT
ga4_total_engagement_sec  BIGINT
sessions_organic          BIGINT
sessions_direct           BIGINT
sessions_referral         BIGINT
sessions_social           BIGINT
sessions_paid             BIGINT
sessions_ai               BIGINT
ai_chatgpt                BIGINT
ai_perplexity             BIGINT
ai_gemini                 BIGINT
ai_copilot                BIGINT
ai_claude                 BIGINT
ai_meta                   BIGINT
ai_other                  BIGINT
scroll_events             BIGINT
month 

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one page, for one client, on one day.

In [21]:


con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) c
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING c > 1
    LIMIT 5
""")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬────────────────┬─────────────────┬───────┐
│ report_date │ client_hash_id │ content_hash_id │   c   │
│    date     │    varchar     │     varchar     │ int64 │
├─────────────┴────────────────┴─────────────────┴───────┤
│                         0 rows                         │
└────────────────────────────────────────────────────────┘

In [22]:
con.sql(f"""
    SELECT COUNT(*) AS row_count, MIN(report_date) AS earliest, MAX(report_date) AS latest
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
""")

┌───────────┬────────────┬────────────┐
│ row_count │  earliest  │   latest   │
│   int64   │    date    │    date    │
├───────────┼────────────┼────────────┤
│   9841378 │ 2026-03-01 │ 2026-03-31 │
└───────────┴────────────┴────────────┘

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Context (IDs/keys — grouping and filtering only, never model input):
report_date, client_hash_id, content_hash_id, month

Context / availability flags (used to filter which rows are usable, not fed to the model):
client_has_gsc, client_has_ga4, gsc_data_available, ga4_data_available

Label / proxy:
No single raw column is the label. A decline label would be computed from a
later time slice of traffic (e.g. ga4_sessions or gsc_clicks) compared to an
earlier slice — same idea as is_declining_label in the starter dataset.

Feature candidates (safe only if measured strictly BEFORE the prediction moment):
gsc_impressions, gsc_clicks, gsc_sum_position, gsc_avg_position, ga4_pageviews,
ga4_sessions, ga4_users, ga4_engaged_sessions, ga4_total_engagement_sec,
sessions_organic/direct/referral/social/paid/ai, ai_chatgpt/perplexity/gemini/
copilot/claude/meta/other, scroll_events

Excluded:
Nothing is excluded outright. The real risk is any feature-candidate column
measured in the SAME window used to define the label — that's the trap Section
3's leakage test is built to catch, not a specific bad column.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [23]:
con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────────┬────────────────────┐
│ total_rows │ gsc_available_rows │ ga4_available_rows │
│   int64    │       int64        │       int64        │
├────────────┼────────────────────┼────────────────────┤
│    9841378 │            3611061 │             413966 │
└────────────┴────────────────────┴────────────────────┘

In [24]:
features_df = con.sql(f"""
    SELECT
        report_date, client_hash_id, content_hash_id,
        gsc_clicks, gsc_avg_position, ga4_engaged_sessions,
        scroll_events, sessions_ai
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available IS TRUE AND ga4_data_available IS TRUE
    LIMIT 1000
""").df()
features_df.head()

,report_date,client_hash_id,content_hash_id,gsc_clicks,gsc_avg_position,ga4_engaged_sessions,scroll_events,sessions_ai
0,2026-03-01,client_65de48885f4ef01b,content_5c80451459c29b4a,0,5.400000,0,0,0
1,2026-03-01,client_65de48885f4ef01b,content_b1f61fc81b28b2d4,0,5.666667,0,0,1
2,2026-03-01,client_65de48885f4ef01b,content_e25ea7297a1dffd3,0,5.156425,0,0,1
3,2026-03-01,client_65de48885f4ef01b,content_6b0149a80607dac3,0,7.694444,0,0,1
4,2026-03-01,client_65de48885f4ef01b,content_62673eea26c31c17,1,6.167885,0,0,0


In [25]:
import numpy as np

features_df['label'] = (features_df['gsc_clicks'] > 0).astype(int)
features_df['leaky_feature'] = features_df['label'] * 100

honest_features = ['gsc_avg_position', 'ga4_engaged_sessions', 'scroll_events', 'sessions_ai']
leaky_features = honest_features + ['leaky_feature']

print("Correlation with label — honest features only:")
print(features_df[honest_features + ['label']].corr()['label'])

print("\nCorrelation with label — WITH the leaky feature added:")
print(features_df[leaky_features + ['label']].corr()['label'])

Correlation with label — honest features only:
gsc_avg_position       -0.069299
ga4_engaged_sessions    0.117790
scroll_events           0.085178
sessions_ai            -0.090398
label                   1.000000
Name: label, dtype: float64

Correlation with label — WITH the leaky feature added:
gsc_avg_position       -0.069299
ga4_engaged_sessions    0.117790
scroll_events           0.085178
sessions_ai            -0.090398
leaky_feature           1.000000
label                   1.000000
Name: label, dtype: float64


In [26]:
features_df = features_df.drop(columns=['leaky_feature'])
print("Leaky feature removed. Keeping the honest feature set only.")


Leaky feature removed. Keeping the honest feature set only.


leaky_feature hit a perfect 1.0 correlation with the label — an obvious sign
it was derived from the answer itself, not a genuine predictor. Removed; the
honest features (weak, realistic correlations between -0.09 and 0.12) are
what's kept.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This slice can't reliably answer questions using GA4 engagement data — only
about 4.2% of March 2026 rows have ga4_data_available = TRUE. GSC search data
is better covered but still incomplete, at about 36.7% of rows. Any feature or
finding built from GA4 columns without filtering on ga4_data_available first
would be conflating "not yet tracked" with "genuinely zero" — a completely
different situation. This isn't random missingness; it follows each client's
own data-start date (per dim_clients.gsc_data_start / ga4_data_start), so the
gap is structural, not noise.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.